In [2]:
# ============================================
# 04_statistical_analysis.ipynb
# Global Inflation Analysis - Statistical Tests
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, f_oneway, pearsonr, spearmanr
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("📊 GLOBAL INFLATION ANALYSIS - STATISTICAL ANALYSIS")
print("=" * 60)

# ============================================
# 1. Load Data (from same folder)
# ============================================

df = pd.read_csv('inflation_cleaned.csv')   # <-- changed from '../data/...'
print(f"✅ Loaded {len(df)} observations from 'inflation_cleaned.csv'")

# ============================================
# 2. Research Questions
# ============================================

print("\n" + "=" * 60)
print("🔍 RESEARCH QUESTIONS")
print("=" * 60)
print("""1. Is there a significant difference in inflation rates between regions?
2. Is there a correlation between inflation and GDP growth?
3. Has global inflation changed significantly over time?
4. Do countries with different income levels have different inflation rates?""")

# ============================================
# 3. ANOVA: Inflation by Region
# ============================================

print("\n" + "=" * 60)
print("📊 ANOVA: INFLATION BY REGION")
print("=" * 60)

# Prepare data
region_groups = [df[df['Region'] == region]['Inflation_Rate_Capped'].dropna()
                 for region in df['Region'].unique() if len(df[df['Region'] == region]) > 30]

# Perform ANOVA
f_stat, p_value = f_oneway(*region_groups)

print(f"\n🔹 One-way ANOVA: Inflation by Region")
print(f"   F-statistic: {f_stat:.4f}")
print(f"   P-value: {p_value:.6f}")

if p_value < 0.05:
    print("   ✅ Result: Statistically significant (p < 0.05)")
    print("   → Inflation rates differ significantly across regions.")
else:
    print("   ❌ Result: Not statistically significant (p >= 0.05)")

# Post-hoc: Pairwise t-tests (Bonferroni correction)
print("\n🔸 Pairwise t-tests (with Bonferroni correction):")
regions = list(df['Region'].unique())
regions = [r for r in regions if len(df[df['Region'] == r]) > 30]

significant_pairs = []
for i in range(len(regions)):
    for j in range(i+1, len(regions)):
        data1 = df[df['Region'] == regions[i]]['Inflation_Rate_Capped'].dropna()
        data2 = df[df['Region'] == regions[j]]['Inflation_Rate_Capped'].dropna()
        t_stat, p_val = ttest_ind(data1, data2)
        # Bonferroni correction
        if p_val < 0.05 / (len(regions) * (len(regions)-1) / 2):
            significant_pairs.append((regions[i], regions[j], p_val))

if significant_pairs:
    print("\n   Significant pairs (after Bonferroni correction):")
    for pair in significant_pairs:
        print(f"   - {pair[0]} vs {pair[1]}: p = {pair[2]:.6f}")
else:
    print("   No significant pairwise differences after correction.")

# ============================================
# 4. Correlation: Inflation vs GDP Growth
# ============================================

print("\n" + "=" * 60)
print("📈 CORRELATION: INFLATION vs GDP GROWTH")
print("=" * 60)

# Remove missing values
corr_data = df[['Inflation_Rate_Capped', 'GDP_Growth']].dropna()

# Pearson correlation (linear)
pearson_r, pearson_p = pearsonr(corr_data['Inflation_Rate_Capped'], corr_data['GDP_Growth'])

# Spearman correlation (non-linear)
spearman_r, spearman_p = spearmanr(corr_data['Inflation_Rate_Capped'], corr_data['GDP_Growth'])

print(f"\n🔹 Pearson Correlation:")
print(f"   Coefficient: {pearson_r:.4f}")
print(f"   P-value: {pearson_p:.6f}")

print(f"\n🔸 Spearman Correlation:")
print(f"   Coefficient: {spearman_r:.4f}")
print(f"   P-value: {spearman_p:.6f}")

if abs(pearson_r) > 0.3 and pearson_p < 0.05:
    print("   → Weak-to-moderate correlation detected.")
    if pearson_r < 0:
        print("   → Inflation and GDP growth tend to move in opposite directions.")
    else:
        print("   → Inflation and GDP growth tend to move in the same direction.")
else:
    print("   → No significant linear correlation detected.")

# ============================================
# 5. Time Series: Has inflation changed over time?
# ============================================

print("\n" + "=" * 60)
print("📅 TIME SERIES ANALYSIS")
print("=" * 60)

# Split data into two periods
df_early = df[df['Year'] < 2008]   # Pre-financial crisis
df_late = df[df['Year'] >= 2010]   # Post-crisis

early_inflation = df_early['Inflation_Rate_Capped'].dropna()
late_inflation = df_late['Inflation_Rate_Capped'].dropna()

# T-test
t_stat, p_value = ttest_ind(early_inflation, late_inflation)

print(f"\n🔹 Comparing Inflation: Pre-2008 vs Post-2010")
print(f"   Pre-2008 mean: {early_inflation.mean():.2f}%")
print(f"   Post-2010 mean: {late_inflation.mean():.2f}%")
print(f"   T-statistic: {t_stat:.4f}")
print(f"   P-value: {p_value:.6f}")

if p_value < 0.05:
    print("   ✅ Result: Statistically significant difference (p < 0.05)")
    print("   → Global inflation patterns have changed significantly.")
else:
    print("   ❌ Result: No significant difference detected.")

# ============================================
# 6. Inflation by Income Level
# ============================================

print("\n" + "=" * 60)
print("💰 INFLATION BY INCOME LEVEL")
print("=" * 60)

income_groups = df.groupby('IncomeLevel')['Inflation_Rate_Capped'].mean().sort_values(ascending=False)
print("\n🔹 Average Inflation by Income Level:")
print(income_groups.round(2))

# ANOVA for income levels
income_levels = [g for g in df['IncomeLevel'].unique() if g != 'Unknown']
income_data = [df[df['IncomeLevel'] == level]['Inflation_Rate_Capped'].dropna()
               for level in income_levels if len(df[df['IncomeLevel'] == level]) > 30]

if len(income_data) >= 2:
    f_stat, p_value = f_oneway(*income_data)
    print(f"\n🔸 ANOVA: Inflation by Income Level")
    print(f"   F-statistic: {f_stat:.4f}")
    print(f"   P-value: {p_value:.6f}")
    if p_value < 0.05:
        print("   ✅ Result: Statistically significant (p < 0.05)")
        print("   → Inflation varies significantly by income level.")
    else:
        print("   ❌ Result: Not statistically significant.")
else:
    print("\n   Not enough income groups with sufficient data for ANOVA.")

# ============================================
# 7. Summary of Findings
# ============================================

print("\n" + "=" * 60)
print("📋 SUMMARY OF STATISTICAL FINDINGS")
print("=" * 60)

# Build dynamic summary based on results (you can adjust thresholds)
region_sig = p_value < 0.05 if 'p_value' in locals() else False
corr_sig = pearson_p < 0.05 and abs(pearson_r) > 0.1
time_sig = p_value < 0.05 if 'p_value' in locals() and 't_stat' in locals() else False
income_sig = p_value < 0.05 if 'p_value' in locals() and 'income_data' in locals() else False

print(f"""
 Key Statistical Findings:

 1. Regional Differences:
    {'✅' if region_sig else '❌'} Inflation rates differ significantly across regions.

 2. Inflation-GDP Relationship:
    {'✅' if corr_sig else '❌'} {'Weak' if abs(pearson_r) < 0.3 else 'Moderate'} correlation detected (r = {pearson_r:.2f}).

 3. Temporal Changes:
    {'✅' if time_sig else '❌'} Significant change in global inflation patterns over time.

 4. Income Level Impact:
    {'✅' if income_sig else '❌'} Inflation varies by country income level.
""")


📊 GLOBAL INFLATION ANALYSIS - STATISTICAL ANALYSIS
✅ Loaded 7418 observations from 'inflation_cleaned.csv'

🔍 RESEARCH QUESTIONS
1. Is there a significant difference in inflation rates between regions?
2. Is there a correlation between inflation and GDP growth?
3. Has global inflation changed significantly over time?
4. Do countries with different income levels have different inflation rates?

📊 ANOVA: INFLATION BY REGION

🔹 One-way ANOVA: Inflation by Region
   F-statistic: 14.0697
   P-value: 0.000000
   ✅ Result: Statistically significant (p < 0.05)
   → Inflation rates differ significantly across regions.

🔸 Pairwise t-tests (with Bonferroni correction):

   Significant pairs (after Bonferroni correction):
   - Other vs Europe: p = 0.001040
   - Other vs South America: p = 0.000000
   - Other vs Africa: p = 0.000007
   - Oceania vs Asia: p = 0.000045
   - Oceania vs South America: p = 0.000275
   - Oceania vs Africa: p = 0.000620
   - Europe vs South America: p = 0.000000
   - Euro